In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('yellow_taxi_processing') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/07 22:17:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
yellow_schema = types.StructType([
    types.StructField("VendorID", types.IntegerType(), True),
    types.StructField("tpep_pickup_datetime", types.TimestampType(), True),
    types.StructField("tpep_dropoff_datetime", types.TimestampType(), True),
    types.StructField("passenger_count", types.IntegerType(), True),
    types.StructField("trip_distance", types.DoubleType(), True),
    types.StructField("RatecodeID", types.IntegerType(), True),
    types.StructField("store_and_fwd_flag", types.StringType(), True),
    types.StructField("PULocationID", types.IntegerType(), True),
    types.StructField("DOLocationID", types.IntegerType(), True),
    types.StructField("payment_type", types.IntegerType(), True),
    types.StructField("fare_amount", types.DoubleType(), True),
    types.StructField("extra", types.DoubleType(), True),
    types.StructField("mta_tax", types.DoubleType(), True),
    types.StructField("tip_amount", types.DoubleType(), True),
    types.StructField("tolls_amount", types.DoubleType(), True),
    types.StructField("improvement_surcharge", types.DoubleType(), True),
    types.StructField("total_amount", types.DoubleType(), True),
    types.StructField("congestion_surcharge", types.DoubleType(), True)
])

In [3]:
input_path = 'data/yellow_tripdata_2025-11.parquet'

# 先不带 schema 读取（此时是 Long），然后立即 cast 回 Integer 
df_yellow_202511 = spark.read.parquet(input_path) \
    .select([F.col(c).cast("int") if "ID" in c or "count" in c else F.col(c) for c in yellow_schema.fieldNames()])


In [4]:
# Repartition to 4 partitions and save to parquet
df = df_yellow_202511.repartition(4)
output_path = 'data/pq/yellow/2025/11/'

df.write.parquet(output_path, mode='overwrite')

In [5]:
# Verify the output
df_verify = spark.read.parquet(output_path)
df_verify.show(5)
print(f"Number of partitions: {df_verify.rdd.getNumPartitions()}")

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|       2| 2025-11-01 21:30:47|  2025-11-01 21:43:13|              1|         2.99|         1|                 N|          87|           4|           1|       16.3|  1.0|    0.5|       1.0|         0.0|                  1.0

In [6]:
from pyspark.sql import functions as F

# Q3: How many taxi trips were there on the 15th of November?
# Filtering by trips that started on 2025-11-15
df_15th = df_verify \
    .withColumn('pickup_date', F.to_date(df_verify.tpep_pickup_datetime)) \
    .filter(F.col('pickup_date') == '2025-11-15')

print(f"Number of taxi trips on November 15th: {df_15th.count()}")

Number of taxi trips on November 15th: 162604


In [7]:
# Q4: What is the length of the longest trip in the dataset in hours?
df_duration = df_verify \
    .withColumn('duration_seconds', 
                F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime')) \
    .withColumn('duration_hours', F.col('duration_seconds') / 3600)

longest_trip = df_duration.select(F.max('duration_hours')).collect()[0][0]
print(f"Longest trip duration in hours: {longest_trip:.1f}")

Longest trip duration in hours: 90.6


In [8]:
# Q6: What is the name of the LEAST frequent pickup location Zone?

# Load zone lookup data
zones_path = 'data/taxi_zone_lookup.csv'
df_zones = spark.read \
    .option("header", "true") \
    .csv(zones_path)

# Register as temp views
df_verify.createOrReplaceTempView('yellow_taxi_data')
df_zones.createOrReplaceTempView('zones')

# Join and find the least frequent pickup location
least_frequent_zone = spark.sql("""
    SELECT 
    z.Zone, 
    COUNT(1) as trip_count
    FROM yellow_taxi_data t
    JOIN zones z ON t.PULocationID = z.LocationID
    WHERE t.tpep_pickup_datetime >= '2025-11-01 00:00:00' 
      AND t.tpep_pickup_datetime < '2025-12-01 00:00:00'
    GROUP BY z.Zone
    ORDER BY trip_count ASC, z.Zone ASC
    LIMIT 5
""").show(truncate=False)

+---------------------------------------------+----------+
|Zone                                         |trip_count|
+---------------------------------------------+----------+
|Arden Heights                                |1         |
|Eltingville/Annadale/Prince's Bay            |1         |
|Governor's Island/Ellis Island/Liberty Island|1         |
|Port Richmond                                |3         |
|Great Kills                                  |4         |
+---------------------------------------------+----------+

